In [0]:
# load the project tables
# Initialize the monitoring notebook
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = "retail_lakehouse"

silver_table = f"{catalog}.{schema_name}.silver_orders"
products_table = f"{catalog}.{schema_name}.bronze_products"
gold_table = f"{catalog}.{schema_name}.gold_daily_sales"

orders = spark.table(silver_table)
products = spark.table(products_table)
gold_sales = spark.table(gold_table)

print("Silver orders:", orders.count())
print("Products:", products.count())
print("Gold daily sales rows:", gold_sales.count())

In [0]:
# Define automated data quality rules
# Create reusable checks

# Check 1: Missing order IDs
missing_order_ids = orders.filter(
    F.col("order_id").isNull()
).count()

# Check 2: Duplicate order IDs
duplicate_order_ids = (
    orders.groupBy("order_id")
    .count()
    .filter(
        F.col("order_id").isNotNull() &
        (F.col("count") > 1)
    )
    .count()
)

# Check 3: Invalid quantities
invalid_quantities = orders.filter(
    F.col("quantity").isNull() |
    (F.col("quantity") <= 0)
).count()

# Check 4: Invalid prices
invalid_prices = orders.filter(
    F.col("unit_price").isNull() |
    (F.col("unit_price") < 0)
).count()

# Check 5: Product IDs missing from the product table
unknown_products = (
    orders.select("product_id").distinct()
    .join(
        products.select("product_id").distinct(),
        on="product_id",
        how="left_anti"
    )
    .count()
)

# Check 6: Negative Gold revenue totals
negative_gold_revenue = gold_sales.filter(
    F.col("total_revenue").isNull() |
    (F.col("total_revenue") < 0)
).count()

print("Missing order IDs:", missing_order_ids)
print("Duplicate order IDs:", duplicate_order_ids)
print("Invalid quantities:", invalid_quantities)
print("Invalid prices:", invalid_prices)
print("Unknown products:", unknown_products)
print("Invalid Gold revenue rows:", negative_gold_revenue)

In [0]:
#  Generate a quality report
# Record the results
check_results = [
    (
        "order_id_completeness",
        missing_order_ids,
        "0 missing order IDs"
    ),
    (
        "order_id_uniqueness",
        duplicate_order_ids,
        "0 duplicate order IDs"
    ),
    (
        "quantity_validity",
        invalid_quantities,
        "0 null or non-positive quantities"
    ),
    (
        "price_validity",
        invalid_prices,
        "0 null or negative prices"
    ),
    (
        "product_referential_integrity",
        unknown_products,
        "0 unknown product IDs"
    ),
    (
        "gold_revenue_validity",
        negative_gold_revenue,
        "0 null or negative revenue rows"
    )
]

quality_rows = [
    (
        check_name,
        int(failed_count),
        "PASS" if failed_count == 0 else "FAIL",
        expected_condition
    )
    for check_name, failed_count, expected_condition
    in check_results
]

quality_report = spark.createDataFrame(
    quality_rows,
    [
        "check_name",
        "failed_records",
        "status",
        "expected_condition"
    ]
).withColumn(
    "run_timestamp",
    F.current_timestamp()
)

display(quality_report)

In [0]:
# Persist the monitoring results
# Create the audit table
monitoring_table = (
    f"{catalog}.{schema_name}.data_quality_audit"
)

(
    quality_report.write
    .format("delta")
    .mode("append")
    .saveAsTable(monitoring_table)
)

print("Quality results saved to:", monitoring_table)

In [0]:
# Check the latest results
# Query the audit table
audit = spark.table(monitoring_table)

latest_run = audit.agg(
    F.max("run_timestamp").alias("latest_timestamp")
).first()["latest_timestamp"]

latest_results = audit.filter(
    F.col("run_timestamp") == latest_run
)

display(
    latest_results.orderBy("check_name")
)

failed_checks = latest_results.filter(
    F.col("status") == "FAIL"
).count()

print("Failed checks in latest run:", failed_checks)

In [0]:
# Validate the monitoring process
# Add assertions
assert latest_results.count() == 6, (
    "Expected six quality checks in the latest run."
)

assert latest_results.filter(
    ~F.col("status").isin("PASS", "FAIL")
).count() == 0, "Unexpected quality status found."

print("Monitoring report validation passed.")